In [4]:
import pandas as pd

In [ ]:
from tensorflow.keras.models import load_model
import pandas as pd
import joblib


NCF_MODEL_PATH = r"D:\MINI PROJECT\Checkpoints\ncfmodel.keras"
ncfmodel = load_model(NCF_MODEL_PATH)

In [ ]:
NCF_MODEL_PATH = r"D:\MINI PROJECT\Checkpoints\ncfmodel.keras"
USER_ENCODER_PATH = r"D:\MINI PROJECT\Checkpoints\user_encoder.pkl"
MOVIE_ENCODER_PATH = r"D:\MINI PROJECT\Checkpoints\movie_encoder.pkl"
EMBEDDINGS_PATH = r"D:\MINI PROJECT\Checkpoints\movie_embeddings.npy"

MOVIES_PATH = r"D:\MINI PROJECT\DATASET\movies_final.csv"
RATINGS_PATH = r"D:\MINI PROJECT\DATASET\MovieLensDataset20M\rating.csv"

# -------------------------------
# LOAD DATA
# -------------------------------
print("Loading data...")
movies = pd.read_csv(MOVIES_PATH)

ratings = pd.read_csv(RATINGS_PATH)

# -------------------------------
# LOAD MODEL & ENCODERS
# -------------------------------
print("Loading NCF model...")
ncfmodel = load_model(NCF_MODEL_PATH)

print("Loading encoders...")
user_encoder = joblib.load(USER_ENCODER_PATH)
movie_encoder = joblib.load(MOVIE_ENCODER_PATH)

Loading data...
Loading NCF model...
Loading encoders...


In [8]:
#Using exponential decay for time decay function

import numpy as np

def time_decay(delta_days, lambda_decay=0.01):
    """
    Exponential time decay
    """
    return np.exp(-lambda_decay * delta_days)


In [9]:
ratings["timestamp"] = pd.to_numeric(
    ratings["timestamp"],
    errors="coerce"
)


In [10]:
import time
import numpy as np

def build_time_aware_user_vector(
    user_id,
    ratings_df,
    embeddings,
    movie_id_to_index,
    lambda_decay=0.01
):
    user_ratings = ratings_df[ratings_df.userId == user_id]

    if user_ratings.empty:
        return None

    now = time.time()
    vectors = []
    weights = []

    for _, row in user_ratings.iterrows():
        movie_id = row.movieId
        timestamp = row.timestamp

        if pd.isna(timestamp):
            continue

        if movie_id not in movie_id_to_index:
            continue

        vec = embeddings[movie_id_to_index[movie_id]]

        # Skip invalid embeddings
        if np.isnan(vec).any():
            continue

        delta_days = (now - timestamp) / (60 * 60 * 24)
        weight = np.exp(-lambda_decay * delta_days)

        vectors.append(weight * vec)
        weights.append(weight)

    if len(vectors) == 0:
        return None

    return np.sum(vectors, axis=0) / np.sum(weights)


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_movies_time_aware_hybrid(
    user_id,
    ncf_model,
    ratings_df,
    movies_df,
    embeddings,
    user_encoder,
    movie_encoder,
    alpha=0.7,
    lambda_decay=0.01,
    top_n=10
):
    """
    Hybrid recommender with time-aware content modeling
    """

    # Encode user
    if user_id not in user_encoder.classes_:
        raise ValueError("User not found")

    user_encoded = user_encoder.transform([user_id])[0]

    # Movies already seen
    seen_movies = set(
        ratings_df[ratings_df.userId == user_id]["movieId"]
    )

    # Candidate movies (known to NCF)
    candidate_movies = [
        m for m in movies_df["movieId"]
        if m not in seen_movies and m in movie_encoder.classes_
    ]

    candidate_encoded = movie_encoder.transform(candidate_movies)

    # ---------------- NCF SCORES ----------------
    user_input = np.full(len(candidate_encoded), user_encoded)

    ncf_scores = ncf_model.predict(
        [user_input, candidate_encoded],
        batch_size=1024,
        verbose=0
    ).flatten()

    # Normalize NCF scores
    ncf_scores = (ncf_scores - ncf_scores.min()) / (np.ptp(ncf_scores) + 1e-8)

    # ---------------- CONTENT (TIME-AWARE) ----------------
    movie_id_to_index = dict(zip(movies_df.movieId, movies_df.index))

    user_vector = build_time_aware_user_vector(
        user_id,
        ratings_df,
        embeddings,
        movie_id_to_index,
        lambda_decay=lambda_decay
    )

    if user_vector is None or np.isnan(user_vector).any():
        liked_indices = [
        movie_id_to_index[mid]
        for mid in seen_movies
        if mid in movie_id_to_index
    ]

    user_vector = embeddings[liked_indices].mean(axis=0)


    candidate_indices = [
        movie_id_to_index[mid] for mid in candidate_movies
    ]

    content_scores = cosine_similarity(
        user_vector.reshape(1, -1),
        embeddings[candidate_indices]
    )[0]

    # Normalize content scores
    content_scores = (content_scores - content_scores.min()) / (np.ptp(content_scores) + 1e-8)

    # ---------------- FINAL HYBRID SCORE ----------------
    final_scores = alpha * ncf_scores + (1 - alpha) * content_scores

    top_indices = np.argsort(final_scores)[::-1][:top_n]
    recommended_ids = [candidate_movies[i] for i in top_indices]

    return movies_df[
        movies_df.movieId.isin(recommended_ids)
    ][["movieId", "title", "genres"]]


In [12]:
ratings["timestamp"] = pd.to_numeric(
    ratings["timestamp"],
    errors="coerce"
)



In [13]:
try:
    embeddings = np.load(EMBEDDINGS_PATH)
    print("Loaded existing embeddings:", embeddings.shape)

    # sanity check
    if embeddings.shape[0] != len(movies):
        raise ValueError("Embedding count mismatch")

except Exception as e:
    print("Rebuilding embeddings due to error:", e)

    text_model = SentenceTransformer("all-MiniLM-L6-v2")

    movies["content_text"] = (
        movies["title"].fillna("") + " " +
        movies["genres"].fillna("") + " " +
        movies["overview"].fillna("")
    )

    embeddings = text_model.encode(
        movies["content_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )

    np.save(EMBEDDINGS_PATH, embeddings)
    print("Saved embeddings:", embeddings.shape)

Loaded existing embeddings: (23138, 384)


In [14]:
time_aware_recs = recommend_movies_time_aware_hybrid(
    user_id=10,
    ncf_model=ncfmodel,
    ratings_df=ratings,
    movies_df=movies,
    embeddings=embeddings,
    user_encoder=user_encoder,
    movie_encoder=movie_encoder,
    alpha=0.7,
    lambda_decay=0.01,
    top_n=10
)

time_aware_recs


,movieId,title,genres
274,296,Pulp Fiction,"thriller, crime, comedy"
291,318,The Shawshank Redemption,"drama, crime"
665,750,Dr. Strangelove or: How I Learned to Stop Worr...,"comedy, war"
1052,1197,The Princess Bride,"adventure, family, fantasy, comedy, romance"
1085,1234,The Sting,"comedy, crime, drama"
1108,1262,The Great Escape,"adventure, drama, war"
2022,2324,Life Is Beautiful,"comedy, drama"
10268,49412,It's a Wonderful World,"comedy, romance, crime, mystery"
16005,91233,Lifted,"family, animation, science fiction, comedy"
16240,92259,The Intouchables,"drama, comedy"
